<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS13/10_Neural_Network_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS13: Multi-Layer Perceptron (MLP) Neural Network - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H
- **Dataset**: `mental_health_burnout_prediction_dataset.csv`

---

### Deskripsi Praktikum:
Praktikum ini berfokus pada pembangunan arsitektur **Artificial Neural Network (ANN)** jenis **Multi-Layer Perceptron (MLP)** menggunakan framework **TensorFlow/Keras** untuk menganalisis risiko kelelahan kerja (*burnout*) pada dataset AntiBurnOut.

Terdapat 2 studi kasus arsitektur jaringan saraf:
1. **Model 1 (Klasifikasi Multi-Kelas):** Memprediksi tingkat risiko burnout (`Burnout_Risk`: *Low*, *Moderate*, *High*) dengan fungsi aktivasi *Softmax* dan loss function *Sparse Categorical Crossentropy*.
2. **Model 2 (Regresi Kontinu):** Mengestimasi indeks burnout (`Burnout_Score`: 0 – 100) menggunakan output neuron linear tunggal dengan loss function *Mean Squared Error (MSE)*.

### Konsep Dasar Multi-Layer Perceptron (MLP):
Setiap neuron pada lapisan tersembunyi (*hidden layer*) melakukan transformasi linear bobot input diikuti oleh fungsi aktivasi non-linear:
$$z_j = \sum_{i=1}^{n} w_{ji} x_i + b_j$$
$$a_j = \text{ReLU}(z_j) = \max(0, z_j)$$

Pada lapisan output klasifikasi multi-kelas (3 kelas), probabilitas dihitung menggunakan fungsi **Softmax**:
$$\hat{y}_c = \frac{e^{z_c}}{\sum_{k=1}^{3} e^{z_k}}$$

Optimisasi dilakukan dengan algoritma **Adam** (Adaptive Moment Estimation) dan regularisasi diterapkan menggunakan **Dropout** untuk mencegah *overfitting*.

In [ ]:
# 1. Import Library Kunci
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_absolute_error, mean_squared_error

# Set random seed untuk reproduksibilitas
np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# 2. Memuat Dataset AntiBurnOut
possible_paths = [
    '../mental_health_burnout_prediction_dataset.csv',
    'mental_health_burnout_prediction_dataset.csv',
    '../../mental_health_burnout_prediction_dataset.csv'
]

csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError("Dataset mental_health_burnout_prediction_dataset.csv tidak ditemukan!")

df = pd.read_csv(csv_path)
print(f"Dataset berhasil dimuat. Dimensi: {df.shape[0]} baris x {df.shape[1]} kolom")
df.head()

In [ ]:
# 3. Pra-pemrosesan Data Khusus Neural Network
# Pilih kombinasi fitur prediktif multidimensi (kesehatan, pekerjaan, gaya hidup)
feature_cols = [
    'Age', 'Work_Hours_Per_Week', 'Sleep_Hours', 'Work_Life_Balance',
    'Job_Satisfaction', 'Anxiety_Score', 'Depression_Score',
    'Mood_Score', 'Physical_Activity_Hours', 'Screen_Time_Hours'
]

# Pastikan kolom target ada
target_clf = 'Burnout_Risk'
target_reg = 'Burnout_Score'

# Buat subset data dan imputasi nilai hilang dengan median
data = df[feature_cols + [target_clf, target_reg]].copy()
for col in feature_cols + [target_reg]:
    data[col] = data[col].fillna(data[col].median())

data[target_clf] = data[target_clf].fillna(data[target_clf].mode()[0])

# Encode target klasifikasi (Low: 0, Moderate: 1, High: 2)
risk_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}
data['Risk_Encoded'] = data[target_clf].map(risk_mapping)

X = data[feature_cols].values
y_clf = data['Risk_Encoded'].values
y_reg = data[target_reg].values

# Split dataset: 70% Train, 15% Validation, 15% Test
X_train, X_temp, y_train_c, y_temp_c, y_train_r, y_temp_r = train_test_split(
    X, y_clf, y_reg, test_size=0.30, random_state=42, stratify=y_clf
)
X_val, X_test, y_val_c, y_test_c, y_val_r, y_test_r = train_test_split(
    X_temp, y_temp_c, y_temp_r, test_size=0.50, random_state=42, stratify=y_temp_c
)

# Skalisasi Standardisasi Fitur (Z-score normalisation)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Ukuran Train     : {X_train_scaled.shape[0]} sampel")
print(f"Ukuran Validation: {X_val_scaled.shape[0]} sampel")
print(f"Ukuran Test      : {X_test_scaled.shape[0]} sampel")

### Bagian A: Pembangunan Model 1 — Klasifikasi Multi-Kelas (`Burnout_Risk`)
Arsitektur jaringan saraf:
- **Input Layer**: 10 neuron (sesuai jumlah fitur)
- **Hidden Layer 1**: 128 neuron, ReLU, Dropout 0.3, Batch Normalization
- **Hidden Layer 2**: 64 neuron, ReLU, Dropout 0.2
- **Hidden Layer 3**: 32 neuron, ReLU
- **Output Layer**: 3 neuron (Low, Moderate, High), Softmax Activation

In [ ]:
# 4. Membangun Model MLP Klasifikasi
clf_model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')
], name="MLP_BurnoutRisk_Classifier")

clf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

clf_model.summary()

# Callback Early Stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Latih Model Klasifikasi
history_clf = clf_model.fit(
    X_train_scaled, y_train_c,
    validation_data=(X_val_scaled, y_val_c),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 5. Evaluasi & Visualisasi Model Klasifikasi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(history_clf.history['loss'], label='Train Loss', color='#2563eb', linewidth=2)
axes[0].plot(history_clf.history['val_loss'], label='Val Loss', color='#dc2626', linestyle='--', linewidth=2)
axes[0].set_title('Kurva Loss Pelatihan vs Validasi (Klasifikasi)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Crossentropy)')
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

# Plot Akurasi
axes[1].plot(history_clf.history['accuracy'], label='Train Accuracy', color='#16a34a', linewidth=2)
axes[1].plot(history_clf.history['val_accuracy'], label='Val Accuracy', color='#f59e0b', linestyle='--', linewidth=2)
axes[1].set_title('Kurva Akurasi Pelatihan vs Validasi (Klasifikasi)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Akurasi')
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()
plt.tight_layout()
plt.show()

# Evaluasi pada Data Uji (Test Set)
y_pred_probs = clf_model.predict(X_test_scaled)
y_pred_c = np.argmax(y_pred_probs, axis=1)

target_names = ['Low', 'Moderate', 'High']
print("=== Laporan Klasifikasi Neural Network MLP ===")
print(classification_report(y_test_c, y_pred_c, target_names=target_names))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - MLP Classification', fontsize=12, fontweight='bold')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

### Bagian B: Pembangunan Model 2 — Regresi Kontinu (`Burnout_Score`)
Arsitektur jaringan saraf:
- **Input Layer**: 10 neuron
- **Hidden Layer 1**: 128 neuron, ReLU
- **Hidden Layer 2**: 64 neuron, ReLU
- **Hidden Layer 3**: 32 neuron, ReLU
- **Output Layer**: 1 neuron linear (prediksi nilai kontinu skor burnout)

In [ ]:
# 6. Membangun Model MLP Regresi
reg_model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
], name="MLP_BurnoutScore_Regressor")

reg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

reg_model.summary()

# Latih Model Regresi
history_reg = reg_model.fit(
    X_train_scaled, y_train_r,
    validation_data=(X_val_scaled, y_val_r),
    epochs=25,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 7. Evaluasi Model Regresi pada Data Uji
y_pred_r = reg_model.predict(X_test_scaled).flatten()

r2 = r2_score(y_test_r, y_pred_r)
mae = mean_absolute_error(y_test_r, y_pred_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))

print(f"Metrik Evaluasi Regresi MLP:")
print(f"- R-squared (R2)  : {r2:.4f}")
print(f"- Mean Abs Error  : {mae:.4f}")
print(f"- Root Mean Sq Err: {rmse:.4f}")

# Visualisasi Scatter Plot Prediksi vs Aktual
plt.figure(figsize=(7, 6))
plt.scatter(y_test_r[:1000], y_pred_r[:1000], alpha=0.3, color='#4f46e5', edgecolors='none', s=25)
plt.plot([0, 100], [0, 100], color='#ef4444', linestyle='--', linewidth=2, label='Garis Ideal y=x')
plt.title(f'Prediksi vs Aktual Skor Burnout (R2 = {r2:.3f})', fontsize=12, fontweight='bold')
plt.xlabel('Skor Aktual')
plt.ylabel('Skor Prediksi')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

### Kesimpulan Praktikum JS13:
1. Arsitektur **Multi-Layer Perceptron (MLP)** dengan Dense Layer, Dropout, dan Batch Normalization terbukti mampu memodelkan relasi non-linear kompleks antara indikator beban kerja dan kesehatan mental.
2. Pada model klasifikasi multi-kelas, penggunaan fungsi aktivasi **Softmax** pada lapisan output mampu menghasilkan distribusi probabilitas yang stabil untuk ketiga tingkatan risiko (*Low, Moderate, High*).
3. Pada model regresi kontinu, neuron output linear mampu mengestimasi skor burnout secara akurat dengan nilai eror yang rendah dan konvergensi cepat.